In [ ]:
%%configure -f
{
  "defaultLakehouse": {
    "name": "DemoLakehouse"
  }
}

# Oracle to Fabric Night Shift

A read-only visualization smoke test for the Oracle data mirrored into `DemoLakehouse`. The left panel uses real sales totals from `DEMO_DW.FACT_SALES`; the right panel renders a deterministic Python galaxy.

In [ ]:
GALAXY_SEED = 26
OUTPUT_PATH = "/lakehouse/default/Files/oracle-to-fabric/oracle_to_fabric_night_shift.png"

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter

CANVAS = "#060810"
PANEL = "#0D1320"
INK = "#F4F7FB"
MUTED = "#8A98AA"
RULE = "#263244"
ORACLE = "#F05A47"
AZURE = "#45A7FF"
FABRIC = "#BE7CFF"


def render_observatory(sales_by_store: pd.DataFrame, output_path: str | None = None, seed: int = 26):
    required = {"store_key", "sales_amount"}
    missing = required.difference(sales_by_store.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")

    data = sales_by_store.sort_values("sales_amount", ascending=True).copy()
    data["store_label"] = data["store_key"].map(lambda value: f"S{int(value):02d}")
    total_sales = float(data["sales_amount"].sum())

    figure = plt.figure(figsize=(16, 9), facecolor=CANVAS)
    grid = figure.add_gridspec(1, 2, width_ratios=(1.15, 1), left=0.06, right=0.96, bottom=0.10, top=0.82, wspace=0.12)
    sales_axis = figure.add_subplot(grid[0], facecolor=PANEL)
    galaxy_axis = figure.add_subplot(grid[1], facecolor=PANEL)

    sales_colors = LinearSegmentedColormap.from_list("oracle_azure", ["#263B59", AZURE, ORACLE])
    bar_colors = sales_colors(np.linspace(0.15, 1.0, len(data)))
    bars = sales_axis.barh(data["store_label"], data["sales_amount"], color=bar_colors, height=0.64)
    maximum = float(data["sales_amount"].max())
    for bar, value in zip(bars, data["sales_amount"]):
        sales_axis.text(value + maximum * 0.018, bar.get_y() + bar.get_height() / 2, f"{value / 1000:,.0f}k", va="center", color=INK, fontsize=9, fontfamily="DejaVu Sans Mono")

    sales_axis.set_xlim(0, maximum * 1.24)
    sales_axis.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value / 1000:,.0f}k"))
    sales_axis.tick_params(axis="x", colors=MUTED, labelsize=9)
    sales_axis.tick_params(axis="y", colors=INK, labelsize=9, length=0)
    sales_axis.grid(axis="x", color=RULE, linewidth=0.8, alpha=0.55)
    sales_axis.set_axisbelow(True)
    for spine in sales_axis.spines.values():
        spine.set_visible(False)
    sales_axis.set_title("MIRRORED SALES BY STORE", loc="left", color=INK, fontsize=15, fontweight="bold", pad=18)
    sales_axis.text(0, 1.01, "Oracle FACT_SALES, aggregated in Spark", transform=sales_axis.transAxes, color=MUTED, fontsize=9)

    rng = np.random.default_rng(seed)
    star_count = 5000
    arm_count = 4
    arm = rng.integers(0, arm_count, star_count)
    radius = rng.power(2.15, star_count)
    angle = arm * (2 * np.pi / arm_count) + radius * 5.2 * np.pi + rng.normal(0, 0.16 + radius * 0.22, star_count)
    x = radius * np.cos(angle) + rng.normal(0, 0.018 + radius * 0.035, star_count)
    y = 0.72 * radius * np.sin(angle) + rng.normal(0, 0.018 + radius * 0.025, star_count)
    sizes = np.clip((1.15 - radius) * 10 + rng.random(star_count) * 4, 0.35, 10)
    galaxy_colors = LinearSegmentedColormap.from_list("fabric_space", [AZURE, FABRIC, "#FFD9A8", ORACLE])

    field_x = rng.uniform(-1.15, 1.15, 360)
    field_y = rng.uniform(-0.95, 0.95, 360)
    galaxy_axis.scatter(field_x, field_y, s=rng.uniform(0.2, 2.0, 360), color=INK, alpha=0.30, linewidths=0)
    galaxy_axis.scatter(x, y, s=sizes, c=1 - radius, cmap=galaxy_colors, alpha=0.72, linewidths=0)
    for size, alpha in ((2500, 0.025), (1200, 0.05), (420, 0.12), (90, 0.85)):
        galaxy_axis.scatter([0], [0], s=size, color="#FFE2B8", alpha=alpha, linewidths=0)

    galaxy_axis.set_xlim(-1.12, 1.12)
    galaxy_axis.set_ylim(-0.90, 0.90)
    galaxy_axis.set_aspect("equal")
    galaxy_axis.set_xticks([])
    galaxy_axis.set_yticks([])
    for spine in galaxy_axis.spines.values():
        spine.set_visible(False)
    galaxy_axis.set_title("PYTHON GALAXY", loc="left", color=INK, fontsize=15, fontweight="bold", pad=18)
    galaxy_axis.text(0, 1.01, f"Deterministic star field, seed {seed}", transform=galaxy_axis.transAxes, color=MUTED, fontsize=9)

    figure.text(0.06, 0.93, "ORACLE TO FABRIC / NIGHT SHIFT", color=INK, fontsize=28, fontweight="bold", family="DejaVu Sans")
    figure.text(0.06, 0.875, f"{len(data)} stores  |  {total_sales:,.0f} mirrored sales amount  |  DemoLakehouse", color=MUTED, fontsize=12, family="DejaVu Sans Mono")
    figure.text(0.06, 0.035, "Source: DEMO_DW.FACT_SALES  |  Snapshot + live Oracle CDC", color=MUTED, fontsize=9, family="DejaVu Sans Mono")

    if output_path:
        target = Path(output_path)
        target.parent.mkdir(parents=True, exist_ok=True)
        figure.savefig(target, dpi=180, facecolor=CANVAS, bbox_inches="tight")

    return figure

In [ ]:
sales_by_store = spark.sql("""
SELECT
  CAST(STORE_KEY AS INT) AS store_key,
  CAST(SUM(SALES_AMOUNT) AS DOUBLE) AS sales_amount
FROM `DemoLakehouse`.`DEMO_DW`.`FACT_SALES`
GROUP BY STORE_KEY
ORDER BY STORE_KEY
""").toPandas()

if len(sales_by_store) != 20:
    raise ValueError(f"Expected 20 stores, found {len(sales_by_store)}")
if sales_by_store["sales_amount"].isna().any():
    raise ValueError("Sales totals contain null values")

print(f"Loaded {len(sales_by_store)} stores from DemoLakehouse")

In [ ]:
figure = render_observatory(sales_by_store, OUTPUT_PATH, GALAXY_SEED)
plt.show()
plt.close(figure)
print(f"Saved hero candidate to {OUTPUT_PATH}")

In [ ]:
if not Path(OUTPUT_PATH).is_file():
    raise FileNotFoundError(OUTPUT_PATH)
if Path(OUTPUT_PATH).stat().st_size < 100_000:
    raise ValueError("Rendered image is unexpectedly small")

print("PYTHON_FUN_VALID=true")
print(f"STORE_COUNT={len(sales_by_store)}")
print(f"IMAGE_BYTES={Path(OUTPUT_PATH).stat().st_size}")